### eNextSUT generator

Run this code to generate (or update) our eNextSUT reference database.
By default, it relies on Exiobase Hybrid version (v3.3.18) but could be potentially arranged to any other SUT in the future. 

As if 5th November 2024, the above-mentioned database is parsed, aggregated in terms of electricity commodity (1 commodity) and activities (EMBER power plants technologies) and electricity production mixes are updated according to EMBER ones for a given year.

Just mind to update your paths in the 'paths.yml' file and to specify the user (your initials) and the desired electricity mix in the first cell, then run all the code

In [1]:
import mario
import yaml
import pandas as pd
from support.ember_remapping import map_ember_to_classification

user = 'LR'   # change this to your username
year = 2023   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

In [2]:
# Parse raw SUT
world = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

In [3]:
# Aggregate electricity commodities and activities to match EMBER
world.aggregate("support/aggregate_ee.xlsx",ignore_nan=True)

nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure treatment (conventional), storage and land application', 'Manure treatment (biogas), storage and land application', 'Forestry, logging and related service activities (02)', 'Fishing, operating of fish hatcheries and fish farms; service activities incidental to fishing (05)', 'Mining of coal and lignite; extraction of peat (10)', 'Extraction of crude petroleum and services related to crude oil extraction, excluding surveying', 'Extraction of natural gas and services related to natural gas extraction, excluding surve

In [4]:
# Parse ember electricity generation data, map to exiobase and get electricity mix for a given year 
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = year,
    mode = 'mix',
)

/Users/lorenzorinaldi/Documents/GitHub/eNextHub/eNextSUT/support/ember_remapping.py:20: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [5]:
#%% Update electricity mixes
z = world.z
s = world.s

for region in world.get_index('Region'):
    print(region,end=' ')
    new_mix = ee_mix.loc[(region,slice(None),slice(None)),'Value'].to_frame().sort_index(axis=0) 
    new_mix.index = new_mix.index.get_level_values(2)
    old_market_share = s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')].sum().sum()
    
    s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')] = new_mix.values*old_market_share # check if commodity electricity is called "Electricity" in aggregation excel file
    # s.loc[:,(region,'Commodity','Electricity')] /= s.loc[:,(region,'Commodity','Electricity')].sum()
    print('done')

z.update(s)

world.update_scenarios('baseline',z=z)
world.reset_to_coefficients('baseline')

BR done
LU done
PT done
NO done
ZA done
WM done
FI done
AU done
MT done
IN done
NL done
SE done
PL done
SK done
CN done
IE done
BG done
ID done
FR done
BE done
LT done
WE done
RO done
WF done
DE done
KR done
CY done
TR done
HU done
JP done
AT done
US done
LV done
CA done
HR done
RU done
IT done
EE done
GB done
WA done
CH done
WL done
ES done
DK done
SI done
GR done
CZ done
MX done


In [6]:
world.to_txt(paths['export'])

Database: to calculate V following matrices are need.
['X'].Trying to calculate dependencies.


In [9]:
ghgs = {
    'Carbon dioxide, fossil (air - Emiss)':1,
    'CH4 (air - Emiss)':29,
    'N2O (air - Emiss)':273
    }

f = world.f.loc[ghgs.keys(),(slice(None),'Commodity','Electricity')]
if isinstance(f,pd.Series):
    f = f.to_frame()

for ghg,gwp in ghgs.items():
    f.loc[ghg,:] *= gwp

f = f.sum(0)*3.6
# f = f.to_frame()   
f 
# f.reset_index(inplace=True)
# f.columns = ['Region','Item','Activity','Value']
# f = f.drop('Item',axis=1)
# f.set_index(['Region','Activity'],inplace=True)
# f = f.unstack()
# f = f.droplevel(0,axis=1)
# f.to_clipboard()


Region  Level      Item       
AT      Commodity  Electricity     224.889724
AU      Commodity  Electricity     728.370635
BE      Commodity  Electricity     185.722586
BG      Commodity  Electricity     424.348434
BR      Commodity  Electricity      96.607004
CA      Commodity  Electricity     142.959825
CH      Commodity  Electricity      35.454706
CN      Commodity  Electricity     787.758161
CY      Commodity  Electricity     731.734649
CZ      Commodity  Electricity     612.811806
DE      Commodity  Electricity     455.453439
DK      Commodity  Electricity     200.747147
EE      Commodity  Electricity     455.022672
ES      Commodity  Electricity     198.245261
FI      Commodity  Electricity     123.074653
FR      Commodity  Electricity     118.609255
GB      Commodity  Electricity     250.952391
GR      Commodity  Electricity     408.383175
HR      Commodity  Electricity     235.944919
HU      Commodity  Electricity     236.704496
ID      Commodity  Electricity     897.318335
IE 

In [25]:
ee_prod = world.s.loc[:,('IT','Commodity','Electricity')]
ee_prod = ee_prod.to_frame()
ee_prod.columns = ['EE_market_share']
ee_prod = ee_prod.query('EE_market_share>0')
ee_prod.sort_values('EE_market_share',ascending=False,inplace=True)
ee_prod


ghgs = {
    'Carbon dioxide, fossil (air - Emiss)':1,
    'CH4 (air - Emiss)':29,
    'N2O (air - Emiss)':273
    }

f = world.f.loc[ghgs.keys(),('IT','Activity',ee_prod.index.get_level_values(2))]
if isinstance(f,pd.Series):
    f = f.to_frame()

for ghg,gwp in ghgs.items():
    f.loc[ghg,:] *= gwp

f = f.sum(0)*3.6
f = f.droplevel(0)
f = f.droplevel(0)
f = f.to_frame()
f.columns = ['gCO2eq/kWh']

ee_prod = ee_prod.join(f)
ee_prod.to_clipboard()